# From a modelgami model to MCMC draws in ArviZ

This notebook takes a model, turns it into an unconstrained log density with its gradient, samples it with BlackJAX's NUTS, and converts the array of unconstrained MCMC samples into structured draws in ArviZ.

The model is the conjugate Bayesian regression from `modelgami.examples`: $y \sim N(X\beta, \sigma^2 I)$ with $\beta \mid \sigma^2 \sim N(m_0, \sigma^2 V_0)$ and $\sigma^2 \sim \text{InvGamma}(a_0, b_0)$. Its posterior is known exactly, so we can check the sampler.

Run `./setup_venv.sh` in this folder once and register the kernel it prints; this notebook uses that kernel ("Python (modelgami)").

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import blackjax
import arviz as az

import modelgami as mg
from modelgami.examples import conjugate_regression as cr

jax.config.update("jax_enable_x64", True)

X, y = cr.simulate_data(jax.random.PRNGKey(0), n=100, p=3)
model = cr.make_model(X, y)
model.params_pattern

## 1. The unconstrained log density and its gradient

BlackJAX samples on $\mathbb{R}^d$, so we use the free representation. The joint density in free form includes the log Jacobian by default, which is what a sampler needs. BlackJAX computes the gradient itself with `jax.grad`; we show it here only to make the point.

In [ ]:
logdensity = model.log_density_fn("joint", params_as="free")
grad_logdensity = jax.grad(logdensity)

x0 = model.flatten_params(model.random_params(jax.random.PRNGKey(1)), to_free=True)
print("dimension:", model.free_dim)
print("log density:", logdensity(x0))
print("gradient:", grad_logdensity(x0))

## 2. NUTS with BlackJAX

We run window adaptation (step size and diagonal mass matrix) separately for each of four chains, then sample. Each chain is one `vmap`ped call.

In [ ]:
n_chains, n_warmup, n_draws = 4, 1000, 1000

def run_chain(key, x_init):
    warmup_key, sample_key = jax.random.split(key)
    warmup = blackjax.window_adaptation(blackjax.nuts, logdensity)
    (state, tuned), _ = warmup.run(warmup_key, x_init, num_steps=n_warmup)
    step = blackjax.nuts(logdensity, **tuned).step

    def one_step(state, key):
        state, info = step(key, state)
        return state, (state.position, info.is_divergent)

    _, (positions, divergent) = jax.lax.scan(one_step, state, jax.random.split(sample_key, n_draws))
    return positions, divergent

init_keys = jax.random.split(jax.random.PRNGKey(2), n_chains)
x_inits = jax.vmap(lambda k: model.flatten_params(model.random_params(k), to_free=True))(init_keys)
chain_keys = jax.random.split(jax.random.PRNGKey(3), n_chains)

free_draws, divergent = jax.jit(jax.vmap(run_chain))(chain_keys, x_inits)
print("free draws:", free_draws.shape, "(chain, draw, free dimension)")
print("divergences:", int(divergent.sum()))

## 3. From unconstrained arrays to structured draws

The pattern folds one free vector into a dict of constrained parameters. Two `vmap`s fold every draw of every chain.

In [ ]:
fold = lambda x: model.fold_params(x, from_free=True)
draws = jax.vmap(jax.vmap(fold))(free_draws)
{k: v.shape for k, v in draws.items()}

## 4. ArviZ

`model.to_arviz` does the folding itself when given free draws, and adds the pointwise log likelihood (for LOO) and the observed data.

In [ ]:
idata = model.to_arviz(free_draws, params_as="free", coords={"beta_dim_0": ["intercept", "x1", "x2"]},
                       dims={"beta": ["beta_dim_0"]})
idata

In [ ]:
az.summary(idata, var_names=["beta", "sigma2"])

In [ ]:
az.loo(idata)

## 5. Checking against the exact posterior

In [ ]:
post = cr.exact_posterior(X, y, model.hyper)
an, bn = post["a0"], post["b0"]
print("posterior mean of beta   exact:", np.round(post["m0"], 4), "  NUTS:", np.round(draws["beta"].mean(axis=(0, 1)), 4))
print("posterior mean of sigma2 exact:", round(float(bn / (an - 1)), 4), "  NUTS:", round(float(draws["sigma2"].mean()), 4))

In [ ]:
import matplotlib.pyplot as plt
from jax.scipy import stats

exact = cr.sample_exact_posterior(jax.random.PRNGKey(4), X, y, model.hyper, n_chains=1, n_draws=20000)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for i in range(3):
    axes[i].hist(np.asarray(draws["beta"][..., i]).ravel(), bins=50, density=True, alpha=0.6, label="NUTS")
    axes[i].hist(np.asarray(exact["beta"][..., i]).ravel(), bins=50, density=True, histtype="step", lw=2, label="exact")
    axes[i].set_title(f"beta[{i}]")
axes[3].hist(np.asarray(draws["sigma2"]).ravel(), bins=50, density=True, alpha=0.6, label="NUTS")
grid = np.linspace(float(draws["sigma2"].min()), float(draws["sigma2"].max()), 200)
axes[3].plot(grid, np.exp(cr.log_inv_gamma(grid, an, bn)), lw=2, label="exact")
axes[3].set_title("sigma2"); axes[3].legend()
plt.tight_layout(); plt.show()